## Загрузка и дубликаты

In [1]:
import pandas as pd

train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
test_original = pd.read_csv("data/test_original.csv")

# Полные дубликаты
datasets = {"Трейн": train, "Тест": test, "Тест оригинал": test_original}
counts = []
for name, frame in datasets.items():
    cleaned = frame.drop_duplicates().reset_index(drop=True)
    counts.append({"Набор": name, "До": len(frame), "Удалено": len(frame) - len(cleaned), "После": len(cleaned)})
    datasets[name] = cleaned
train, test, test_original = (datasets[name] for name in ("Трейн", "Тест", "Тест оригинал"))
display(pd.DataFrame(counts).set_index("Набор"))


,До,Удалено,После
Набор,,,
Трейн,891,0,891
Тест,1000,0,1000
Тест оригинал,418,0,418


## Сравнение тестов

In [2]:
# Общие признаки
common_features = [column for column in test.columns
                  if column in test_original.columns and column not in ["Id", "PassengerId", "Survived"]]
print("Признаки для сравнения:", common_features)
print("Только в тесте:", sorted(set(test.columns) - set(test_original.columns)))
print("Только в оригинале:", sorted(set(test_original.columns) - set(test.columns)))

# Совпадения строк
left = test[common_features]
right = test_original[common_features]
left_matches = left.merge(right.drop_duplicates(), on=common_features, how="left", indicator=True)["_merge"].eq("both")
right_matches = right.merge(left.drop_duplicates(), on=common_features, how="left", indicator=True)["_merge"].eq("both")
unique_left = left.drop_duplicates()
unique_right = right.drop_duplicates()
shared_unique = unique_left.merge(unique_right, on=common_features, how="inner")
comparison = pd.DataFrame([
    {"Набор": "Тест", "Строк": len(left), "Совпало": int(left_matches.sum()), "Совпадение, %": 100 * left_matches.mean()},
    {"Набор": "Тест оригинал", "Строк": len(right), "Совпало": int(right_matches.sum()), "Совпадение, %": 100 * right_matches.mean()},
]).set_index("Набор")
display(comparison.round(2))
print("Уникальных комбинаций в тесте:", len(unique_left))
print("Уникальных комбинаций в оригинале:", len(unique_right))
print("Общих уникальных комбинаций:", len(shared_unique))
print("Значения и порядок строк совпадают:", left.equals(right))


Признаки для сравнения: ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
Только в тесте: ['Id']
Только в оригинале: ['Cabin', 'Name', 'PassengerId', 'Ticket']


,Строк,Совпало,"Совпадение, %"
Набор,,,
Тест,1000,7,0.70
Тест оригинал,418,7,1.67


Уникальных комбинаций в тесте: 1000
Уникальных комбинаций в оригинале: 380
Общих уникальных комбинаций: 7
Значения и порядок строк совпадают: False


In [3]:
# Совпавшие строки
matched_rows = test.merge(test_original, on=common_features, how="inner")
display(matched_rows[["Id", "PassengerId"] + common_features + ["Name", "Ticket", "Cabin"]])


,Id,PassengerId,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Name,Ticket,Cabin
0,13977915,1221,2,male,21.0,0,0,13.0000,S,"Enander, Mr. Ingvar",236854,NaN
1,20017851,1220,2,male,29.0,1,0,26.0000,S,"Clarke, Mr. Charles Valentine",2003,NaN
2,66478563,1215,1,male,33.0,0,0,26.5500,S,"Rowe, Mr. Alfred G",113790,NaN
3,19633265,1015,3,male,28.0,0,0,7.2500,S,"Carver, Mr. Alfred John",392095,NaN
4,70078256,1027,3,male,24.0,0,0,7.8542,S,"Carlsson, Mr. Carl Robert",350409,NaN
5,77178428,1192,3,male,32.0,0,0,7.7750,S,"Olsson, Mr. Oscar Wilhelm",347079,NaN
6,37747596,909,3,male,21.0,0,0,7.2250,C,"Assaf, Mr. Gerios",2692,NaN
